# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [2]:
%pip install duckdb polars pyarrow

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.1 MB 837.5 kB/s eta 0:00:19
   - -------------------------------------- 0.8/16.1 MB 1.0 MB/s eta 0:00:16
   -- ------------------------------------- 1.0/16.1 MB 1.1 MB/s eta 0:00:14
   --- ------------------------------------ 1.6/16.1 MB 1.3 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/16.1 MB 1.2 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/16.1 MB 1.2 MB/s eta 0:00:12
   ----- ---------------------------------- 2.1/16.1 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.1/16.1 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.4/16.1 MB 1.0 MB/s eta 0:00:14
   ------- --------------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nIsi folder saat ini:")
for item in Path.cwd().iterdir():
    print(item.name)

Current working directory:
c:\Users\NISA\tugas1-abd-2026\notebooks

Isi folder saat ini:
01_data_profiling.ipynb
02_data_cleaning.ipynb
03_eda_and_insights.ipynb


In [ ]:
from pathlib import Path

import duckdb
import polars as pl
PROJECT_ROOT = Path.cwd().parent.resolve()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "globalLandTemperaturesByCity.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}\n"
        "Pastikan file berada di folder data/raw."
    )

print("Dataset berhasil ditemukan!")
print(f"Nama file : {DATA_PATH.name}")
print(f"Lokasi   : {DATA_PATH}")
print(f"Ukuran   : {DATA_PATH.stat().st_size / 1_000_000:.2f} MB")

Dataset berhasil ditemukan!
Nama file : globalLandTemperaturesByCity.csv
Lokasi   : C:\Users\NISA\tugas1-abd-2026\data\raw\globalLandTemperaturesByCity.csv
Ukuran   : 27.11 MB


In [14]:
print("Project root:", PROJECT_ROOT)
print("Folder data:", PROJECT_ROOT / "data")
print("Folder raw :", PROJECT_ROOT / "data" / "raw")
print("File CSV  :", DATA_PATH)
print("File ada  :", DATA_PATH.exists())

Project root: C:\Users\NISA\tugas1-abd-2026
Folder data: C:\Users\NISA\tugas1-abd-2026\data
Folder raw : C:\Users\NISA\tugas1-abd-2026\data\raw
File CSV  : C:\Users\NISA\tugas1-abd-2026\data\raw\globalLandTemperaturesByCity.csv
File ada  : True


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_csv(DATA_PATH, infer_schema_length=10000)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias("jumlah_baris"),
    pl.all().null_count().name.suffix("_null"),
]).collect()

# Menampilkan hasil ringkasan
ringkasan

Schema({'1951-01-01': String, '-0.311': Float64, '0.486': Float64, 'Århus': String, 'Denmark': String, '57.05N': String, '10.33E': String})


jumlah_baris,1951-01-01_null,-0.311_null,0.486_null,Århus_null,Denmark_null,57.05N_null,10.33E_null
u32,u32,u32,u32,u32,u32,u32,u32
457044,0,0,0,0,0,1,1


In [ ]:
lazy_data.head(5).collect()

1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,f64,f64,str,str,str,str
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-06-01""",14.99402,0.622,"""Århus""","""Denmark""","""57.05N""","""10.33E"""


In [ ]:
df = lazy_data.collect()

print("Jumlah baris :", df.height)
print("Jumlah kolom :", df.width)
print("Nama kolom   :", df.columns)

Jumlah baris : 457044
Jumlah kolom : 7
Nama kolom   : ['1951-01-01', '-0.311', '0.486', 'Århus', 'Denmark', '57.05N', '10.33E']


In [19]:
contoh_data = lazy_data.head(10).collect()
contoh_data

1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,f64,f64,str,str,str,str
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-06-01""",14.99402,0.622,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-07-01""",16.05102,0.488,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-08-01""",16.511,0.204,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-09-01""",14.103,0.378,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-10-01""",8.878,0.452,"""Århus""","""Denmark""","""57.05N""","""10.33E"""


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
import duckdb
import polars as pl
df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True,
    truncate_ragged_lines=True
)

print("Dataset berhasil dibaca!")
print("Jumlah baris :", df.height)
print("Jumlah kolom :", df.width)
print("Nama kolom   :", df.columns)

connection = duckdb.connect()

# Mendaftarkan DataFrame Polars ke DuckDB
connection.register("dataset_df", df)
hasil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset_df"
).fetchall()

nama_kolom = [
    item[0] for item in connection.description
]

profil_duckdb = pl.DataFrame(
    hasil,
    schema=nama_kolom,
    orient="row"
)
display(profil_duckdb)

Dataset berhasil dibaca!
Jumlah baris : 457044
Jumlah kolom : 7
Nama kolom   : ['1951-01-01', '-0.311', '0.486', 'Århus', 'Denmark', '57.05N', '10.33E']


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""1951-01-01""","""VARCHAR""","""1951-01-01""","""1999-12-01""",301,null,null,null,null,null,457044,0.00
"""-0.311""","""DOUBLE""","""-32.086999999999996""","""38.389""",56942,"""18.47733086415183""","""9.7058294423066""","""13.101660300647575""","""20.84702552855228""","""26.01006503442021""",457044,0.00
"""0.486""","""DOUBLE""","""0.04""","""4.1419999999999995""",1898,"""0.36692528357445264""","""0.1919774992497935""","""0.2418339975521282""","""0.32370281277072743""","""0.4390666654728489""",457044,0.00
"""Århus""","""VARCHAR""","""A Coruña""","""Ürümqi""",1565,null,null,null,null,null,457044,0.00
"""Denmark""","""VARCHAR""","""Afghanistan""","""Zimbabwe""",124,null,null,null,null,null,457044,0.00
"""57.05N""","""VARCHAR""","""0.80N""","""8.84S""",68,null,null,null,null,null,457044,0.00
"""10.33E""","""VARCHAR""","""0.00W""","""99.91E""",741,null,null,null,null,null,457044,0.00


In [ ]:
missing_values = df.null_count()

display(missing_values)

1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,1,1


In [ ]:
import duckdb
import polars as pl
df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True,
    truncate_ragged_lines=True
)

connection = duckdb.connect()

connection.register("dataset_csv", df)
query = """
SUMMARIZE SELECT * FROM dataset_csv
"""

result = connection.execute(query)
hasil = result.fetchall()
nama_kolom = [desc[0] for desc in result.description]

df_duckdb = pl.DataFrame(
    hasil,
    schema=nama_kolom,
    orient="row"
)
print("Jumlah baris :", df.height)
print("Jumlah kolom :", df.width)

print("\nNama kolom:")
print(df.columns)
display(df_duckdb)

Jumlah baris : 457044
Jumlah kolom : 7

Nama kolom:
['1951-01-01', '-0.311', '0.486', 'Århus', 'Denmark', '57.05N', '10.33E']


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""1951-01-01""","""VARCHAR""","""1951-01-01""","""1999-12-01""",301,null,null,null,null,null,457044,0.00
"""-0.311""","""DOUBLE""","""-32.086999999999996""","""38.389""",56942,"""18.477330864151863""","""9.705829442306596""","""13.100193783642322""","""20.852243322715882""","""26.00318699644554""",457044,0.00
"""0.486""","""DOUBLE""","""0.04""","""4.1419999999999995""",1898,"""0.3669252835744512""","""0.19197749924979352""","""0.24205693276191695""","""0.32348979380588966""","""0.43904654741962884""",457044,0.00
"""Århus""","""VARCHAR""","""A Coruña""","""Ürümqi""",1565,null,null,null,null,null,457044,0.00
"""Denmark""","""VARCHAR""","""Afghanistan""","""Zimbabwe""",124,null,null,null,null,null,457044,0.00
"""57.05N""","""VARCHAR""","""0.80N""","""8.84S""",68,null,null,null,null,null,457044,0.00
"""10.33E""","""VARCHAR""","""0.00W""","""99.91E""",741,null,null,null,null,null,457044,0.00


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

In [ ]:
# 1. Ukuran dataset
print("=" * 50)
print("UKURAN DATASET")
print("=" * 50)

print(f"Jumlah baris     : {df.height:,}")
print(f"Jumlah kolom     : {df.width}")
print(f"Ukuran file     : {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

# 2. Tipe data kolom
print("\n" + "=" * 50)
print("TIPE DATA KOLOM")
print("=" * 50)

print(df.schema)

# 3. Missing values
print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)

null_summary = pl.DataFrame({
    "Kolom": df.columns,
    "Jumlah_Null": [df[c].null_count() for c in df.columns],
    "Persentase_Null": [
        round(df[c].null_count() / df.height * 100, 2)
        if df.height > 0 else 0
        for c in df.columns
    ]
}).sort("Jumlah_Null", descending=True)

display(null_summary)

# 4. Duplikasi data
print("\n" + "=" * 50)
print("DUPLIKASI DATA")
print("=" * 50)

duplicate_count = df.height - df.unique().height

print(f"Jumlah baris duplikat: {duplicate_count:,}")

# 5. Statistik deskriptif
print("\n" + "=" * 50)
print("STATISTIK DESKRIPTIF")
print("=" * 50)

display(df.describe())

# 6. Menampilkan data awal
print("\n" + "=" * 50)
print("5 DATA PERTAMA")
print("=" * 50)

display(df.head(5))

UKURAN DATASET
Jumlah baris     : 457,044
Jumlah kolom     : 7
Ukuran file     : 25.85 MB

TIPE DATA KOLOM
Schema({'1951-01-01': String, '-0.311': Float64, '0.486': Float64, 'Århus': String, 'Denmark': String, '57.05N': String, '10.33E': String})

MISSING VALUES


Kolom,Jumlah_Null,Persentase_Null
str,i64,f64
"""57.05N""",1,0.0
"""10.33E""",1,0.0
"""1951-01-01""",0,0.0
"""-0.311""",0,0.0
"""0.486""",0,0.0
"""Århus""",0,0.0
"""Denmark""",0,0.0



DUPLIKASI DATA
Jumlah baris duplikat: 0

STATISTIK DESKRIPTIF


statistic,1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,str,f64,f64,str,str,str,str
"""count""","""457044""",457044.0,457044.0,"""457044""","""457044""","""457043""","""457043"""
"""null_count""","""0""",0.0,0.0,"""0""","""0""","""1""","""1"""
"""mean""",null,18.477331,0.366925,null,null,null,null
"""std""",null,9.705829,0.191977,null,null,null,null
"""min""","""1951-01-01""",-32.087,0.04,"""A Coruña""","""Afghanistan""","""0.80N""","""0.00W"""
"""25%""",null,13.097,0.242,null,null,null,null
"""50%""",null,20.846,0.324,null,null,null,null
"""75%""",null,26.006,0.439,null,null,null,null
"""max""","""1999-12-01""",38.389,4.142,"""Ürümqi""","""Zimbabwe""","""8.84S""","""99.91E"""



5 DATA PERTAMA


1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,f64,f64,str,str,str,str
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-06-01""",14.99402,0.622,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
